<a href="https://colab.research.google.com/github/ergul13/mr_akgul/blob/main/IMAGENET1K_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/Lokal_Veriseti
!unzip -q /content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Goruntuler_Saglam.zip -d /content/Lokal_Veriseti/

In [ ]:
import os
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

def hizli_harita_olustur(csv1, csv2, resim_klasoru):
    df1 = pd.read_excel(csv1)
    df2 = pd.read_excel(csv2)
    df_tumu = pd.concat([df1, df2], ignore_index=True)

    df_tumu = df_tumu[df_tumu['BI-RADS 0/1/2/4/5'].str.contains('1|2|4|5', na=False)].copy()

    etiketler = {}
    for _, row in df_tumu.iterrows():
        vaka_no = str(int(row['CASENUMBER']))
        orijinal_etiket = int(re.search(r'\d+', str(row['BI-RADS 0/1/2/4/5'])).group())
        etiketler[vaka_no] = orijinal_etiket

    liste = []

    for kok_dizin, _, dosyalar in os.walk(resim_klasoru):
        for dosya in dosyalar:
            sayilar = re.findall(r'\d{9}', dosya)
            if sayilar:
                vaka = sayilar[0]
                if vaka in etiketler:
                    liste.append({
                        'resim_yolu': os.path.join(kok_dizin, dosya),
                        'etiket': etiketler[vaka]
                    })

    return pd.DataFrame(liste)

class MemeKanseriDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.etiket_donusum = {1: 0, 2: 1, 4: 2, 5: 3}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['resim_yolu']
        image = Image.open(img_path).convert('RGB')

        orijinal_etiket = self.df.iloc[idx]['etiket']
        label = self.etiket_donusum[orijinal_etiket]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

class Hybrid_MobileNet_ViT(nn.Module):
    def __init__(self, num_classes):
        super(Hybrid_MobileNet_ViT, self).__init__()

        self.mobilenet = models.mobilenet_v3_large(weights='IMAGENET1K_V1')
        self.mobilenet.classifier = nn.Identity()

        self.vit = models.vit_b_16(weights='IMAGENET1K_V1')
        self.vit.heads = nn.Identity()

        self.fc = nn.Sequential(
            nn.Linear(960 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        mn_features = self.mobilenet(x)
        vit_features = self.vit(x)

        combined = torch.cat((mn_features, vit_features), dim=1)
        output = self.fc(combined)

        return output

csv1_yolu = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Cikarilan_Veriler/Bilgi/Supplementary_TRAIN1.xlsx'
csv2_yolu = '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Cikarilan_Veriler/Bilgi/Supplementary_TRAIN2.xlsx'
resim_klasoru = '/content/Lokal_Veriseti/'

print("Veri haritalama yapiliyor...")
df_egitim = hizli_harita_olustur(csv1_yolu, csv2_yolu, resim_klasoru)
print(f"Eslesen resim sayisi: {len(df_egitim)}")

if len(df_egitim) > 0:
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = MemeKanseriDataset(df_egitim, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Kullanilan donanim: {device}")

    model = Hybrid_MobileNet_ViT(num_classes=4).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    epochs = 5
    print("Egitim basliyor...")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if (i + 1) % 5 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Adim [{i+1}/{len(train_loader)}] | Anlik Loss: {loss.item():.4f}")

        print(f"--- Epoch {epoch+1} Bitti | Ortalama Loss: {running_loss/len(train_loader):.4f} ---")
else:
    print("Veri bulunamadı. Lütfen dosya yollarını kontrol et.")

Veri haritalama yapiliyor...
Eslesen resim sayisi: 0
Veri bulunamadı. Lütfen dosya yollarını kontrol et.


In [ ]:
!mkdir -p /content/Lokal_Veriseti
!find '/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Saglik_Bakanligi_Kirpilmis' -type f -name "*.png" | xargs -I {} cp {} /content/Lokal_Veriseti/

find: ‘/content/drive/MyDrive/Meme_Kanseri_Proje_Verileri/Saglik_Bakanligi_Kirpilmis’: Input/output error
